## Fase 1: Preparación del Corpus y Línea Base RAG Estándar

In [1]:
!pip install -q langchain langchain-core langchain-community langchain-text-splitters \
langchain-groq langchain-huggingface langchain-chroma chromadb sentence-transformers \
pypdf ragas gradio scipy striprtf langgraph langchain-google-genai
print("Dependencias instaladas")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 119.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 96.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/7

In [2]:
from google.colab import drive
drive.mount('/content/drive')

print("Drive montado correctamente")

Mounted at /content/drive
Drive montado correctamente


In [3]:
import os
import re
import shutil
import logging
import chromadb
import pandas as pd
import time

from pathlib import Path
from typing import TypedDict
from tqdm.notebook import tqdm
from google.colab import userdata

from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from langgraph.graph import StateGraph, END

logging.getLogger("pypdf").setLevel(logging.ERROR)

print("Imports OK")

/tmp/ipykernel_701/2608852746.py:15: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Imports OK


In [4]:
CORPUS_PATH = "/content/drive/MyDrive/TG_Maestria/01_Corpus_Raw/"
VECTOR_LOCAL = "/content/vectorstore/"
VECTOR_DRIVE = "/content/drive/MyDrive/TG_Maestria/02_Vectorstore/"
RESULTS_PATH = "/content/drive/MyDrive/TG_Maestria/03_Results/"

for p in [VECTOR_LOCAL, VECTOR_DRIVE, RESULTS_PATH]:
    os.makedirs(p, exist_ok=True)

print("Rutas configuradas")
print("CORPUS_PATH:", CORPUS_PATH)

Rutas configuradas
CORPUS_PATH: /content/drive/MyDrive/TG_Maestria/01_Corpus_Raw/


In [5]:
emb = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-large",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True}
)

print("Embeddings listos")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

Embeddings listos


In [6]:
CARPETAS_SENTENCIAS = {
    "Sentencias", "sentencias",
    "Jurisprudencia", "jurisprudencia",
    "Corte_Constitucional", "corte_constitucional"
}

def limpiar_norma_markdown(texto: str) -> str:
    texto = texto.replace("\ufeff", "")
    texto = re.sub(r"\n{3,}", "\n\n", texto)
    texto = re.sub(r"[ \t]+", " ", texto)
    return texto.strip()

def limpiar_sentencia(texto: str) -> str:
    patrones_inicio = [
        r"\bCONSIDERACIONES\b",
        r"\bCONSIDERACIÓN\b",
        r"\bFUNDAMENTOS\b",
        r"\bDE LA DECISIÓN\b",
        r"\bCONSIDERACIONES DE LA CORTE\b"
    ]

    inicio_idx = None
    for patron in patrones_inicio:
        m = re.search(patron, texto, flags=re.IGNORECASE)
        if m:
            inicio_idx = m.start()
            break

    if inicio_idx is None:
        inicio_idx = 0

    fin_idx = len(texto)
    mfin = re.search(
        r"Cópiese|Notifíquese|Devuélvase|CÚMPLASE|Cúmplase",
        texto[inicio_idx:],
        flags=re.IGNORECASE
    )

    if mfin:
        fin_idx = inicio_idx + mfin.end() + 500

    texto = texto[inicio_idx:fin_idx]
    texto = re.sub(r"Firmado electrónicamente.*?(?=[A-ZÁÉÍÓÚÑ]{3,}|$)", "", texto, flags=re.DOTALL | re.IGNORECASE)
    texto = re.sub(r"Este documento fue firmado electrónicamente.*", "", texto, flags=re.DOTALL | re.IGNORECASE)
    texto = re.sub(r"Radicación\s*N[°ºo.]?\s*[-:A-Za-z0-9]+", "", texto, flags=re.IGNORECASE)
    texto = re.sub(r"^\s*\d{1,3}\s*$", "", texto, flags=re.MULTILINE)
    texto = re.sub(r"\n{3,}", "\n\n", texto)

    return texto.strip()

def extraer_ley_desde_nombre(nombre_archivo: str) -> str:
    m = re.search(r"ley[_\-\s]?(\d+)", nombre_archivo.lower())
    return m.group(1) if m else "desconocida"

def parse_markdown_articulos(texto: str, fuente: str, categoria: str):
    texto = limpiar_norma_markdown(texto)
    partes = re.split(r"(?m)^##\s+Artículo\s+(\d+)\s*$", texto)
    docs = []

    if len(partes) < 3:
        return docs

    encabezado_general = partes[0].strip()
    ley = extraer_ley_desde_nombre(Path(fuente).stem)

    for i in range(1, len(partes), 2):
        num_articulo = partes[i].strip()
        bloque = partes[i + 1].strip()

        bloque = re.sub(r"(?m)^---\s*$", "", bloque).strip()
        lineas = [l.strip() for l in bloque.splitlines() if l.strip()]
        if not lineas:
            continue

        encabezado_articulo = lineas[0]
        contenido = "\n".join(lineas)

        texto_doc = f"Artículo {num_articulo}\n{contenido}"
        if encabezado_general:
            texto_doc = f"{encabezado_general}\n\n{texto_doc}"

        metadata = {
            "tipo": "norma",
            "formato": "md",
            "categoria": categoria,
            "fuente": Path(fuente).stem,
            "archivo": Path(fuente).name,
            "ley": ley,
            "num_articulo": num_articulo,
            "encabezado": encabezado_articulo
        }

        docs.append(
            Document(
                page_content="passage: " + texto_doc,
                metadata=metadata
            )
        )

    return docs

print("Funciones de limpieza y parser listas")

Funciones de limpieza y parser listas


In [7]:
def detectar_categoria_y_tipo(archivo: Path):
    partes = [p.name for p in archivo.parents]

    if "Corte_Constitucional" in partes:
        categoria = "Corte_Constitucional"
    elif "Sentencias" in partes:
        categoria = "Sentencias"
    elif "sentencias" in partes:
        categoria = "sentencias"
    elif "Jurisprudencia" in partes:
        categoria = "Jurisprudencia"
    elif "jurisprudencia" in partes:
        categoria = "jurisprudencia"
    else:
        categoria = archivo.parent.name

    tipo = "sentencia" if any(p in CARPETAS_SENTENCIAS for p in partes) else "norma"
    return categoria, tipo

def extraer_texto_rtf(ruta_rtf: str) -> str:
    from striprtf.striprtf import rtf_to_text

    with open(ruta_rtf, "r", encoding="utf-8", errors="ignore") as f:
        contenido = f.read()

    texto = rtf_to_text(contenido)
    texto = re.sub(r"\n{3,}", "\n\n", texto)
    return texto.strip()

print("Funciones para RTF listas")

Funciones para RTF listas


In [ ]:
# =============================================================================
# CARGA DEL CORPUS LIMPIO  (reemplaza a cargar_corpus_mixto)
#
# El loader anterior hacia rglob("*") sobre 01_Corpus_Raw y tomaba todo lo que
# encontrara. Hoy esa carpeta guarda tres copias del mismo corpus:
#
#   Corte_Constitucional/archivos/      411 .rtf  -> la SPA de error, inservibles
#   Corte_Constitucional/archivos_v3/   411 doc   -> descarga buena, formato mixto
#   Corte_Constitucional/texto_plano/   411 .txt  -> normalizado, ESTE es el bueno
#   Corte_Suprema/archivos/             301 .pdf  -> 53 corruptos, 143 sin OCR
#   Corte_Suprema/texto_plano/          245 .txt  -> normalizado, ESTE es el bueno
#
# Correr el loader viejo reindexaria el CSS de Bootstrap y triplicaria
# documentos. Este lee UNICAMENTE texto_plano/ y las 4 normas .md.
#
# Se conserva igual: prefijo "passage:", limpiar_sentencia(), el esquema de
# metadata y el splitter, para que los resultados sigan siendo comparables.
# =============================================================================

from pathlib import Path
from langchain_core.documents import Document

CORPUS = Path(CORPUS_PATH)

CARPETAS_TEXTO = [
    CORPUS / "Sentencias" / "Corte_Suprema" / "texto_plano",
    CORPUS / "Sentencias" / "Corte_Constitucional" / "texto_plano",
]

# marcas de que algo del corpus roto se colo pese a todo
CENTINELAS = [
    "--bs-blue", "data-beasties-container", "<style", "sharethis-js",
    "DOCUMENTO NO DISPONIBLE", "font-family:",
]


def _categoria_desde_ruta(archivo: Path) -> str:
    partes = {p.name for p in archivo.parents}
    if "Corte_Constitucional" in partes:
        return "Corte_Constitucional"
    if "Corte_Suprema" in partes:
        return "Sentencias"
    return archivo.parent.name


def cargar_corpus_limpio(verbose: bool = True):
    docs = []
    stats = {"normas": 0, "articulos": 0, "sentencias": 0,
             "vacias": 0, "contaminadas": 0}

    # ---------- 1) normas: los 4 markdown ----------
    for md in sorted(CORPUS.glob("*/*.md")):
        texto = md.read_text(encoding="utf-8")
        arts = parse_markdown_articulos(texto, str(md), md.parent.name)
        docs.extend(arts)
        stats["normas"] += 1
        stats["articulos"] += len(arts)
        if verbose:
            print(f"[MD ] {md.parent.name:<22} {md.name:<30} {len(arts)} articulos")

    # ---------- 2) sentencias: solo texto_plano ----------
    for carpeta in CARPETAS_TEXTO:
        if not carpeta.is_dir():
            print(f"AVISO: no existe {carpeta}")
            continue

        archivos = sorted(carpeta.glob("*.txt"))
        categoria = _categoria_desde_ruta(archivos[0]) if archivos else "?"
        n_ok = 0

        for f in archivos:
            texto = f.read_text(encoding="utf-8", errors="ignore")

            sucio = [c for c in CENTINELAS if c in texto[:4000]]
            if sucio:
                stats["contaminadas"] += 1
                print(f"  DESCARTADO {f.name[:45]} -> contiene {sucio[0]}")
                continue

            texto = limpiar_sentencia(texto).strip()
            if len(texto) <= 30:
                stats["vacias"] += 1
                continue

            docs.append(Document(
                page_content="passage: " + texto,
                metadata={
                    "categoria": categoria,
                    "fuente": f.stem,
                    "archivo": f.name,
                    "formato": "txt",
                    "tipo": "sentencia",
                    "page": 0,
                },
            ))
            n_ok += 1

        stats["sentencias"] += n_ok
        if verbose:
            print(f"[TXT] {categoria:<22} {carpeta.parent.name:<30} {n_ok} sentencias")

    print("\n" + "=" * 60)
    print(f"Normas          : {stats['normas']} archivos -> {stats['articulos']} articulos")
    print(f"Sentencias      : {stats['sentencias']}")
    print(f"Descartadas     : {stats['contaminadas']} contaminadas, {stats['vacias']} vacias")
    print(f"TOTAL documentos: {len(docs)}")
    print("=" * 60)

    if stats["contaminadas"]:
        raise RuntimeError(
            f"{stats['contaminadas']} documentos traian marcas del corpus roto. "
            "Revisa texto_plano/ antes de indexar."
        )
    return docs


print("Loader limpio definido")


In [ ]:
# Prueba de carga: verifica que se lean los 656 documentos antes de gastar GPU
docs_test = cargar_corpus_limpio()

print(f"\nTotal de documentos cargados: {len(docs_test)}")

import collections
print("\nPor categoria:")
for k, v in collections.Counter(d.metadata["categoria"] for d in docs_test).most_common():
    print(f"   {k:<24} {v:6d}")

print("\nMuestra:")
for d in docs_test[:2]:
    print(f"\n  {d.metadata['archivo'][:60]}")
    print(f"  {d.page_content[:220]}")


In [ ]:
# =============================================================================
# CONSTRUCCION DEL VECTORSTORE  (reemplaza a build_vectorstore / load_vectorstore)
#
# Mismo splitter que antes (900 / 120) para mantener comparabilidad.
#
# OJO PARA LA METODOLOGIA: antes los PDF entraban pagina por pagina, asi que
# limpiar_sentencia() se aplicaba a cada pagina aislada y casi nunca hallaba el
# par CONSIDERACIONES -> Copiese. Ahora cada sentencia entra completa y el
# recorte SI opera: de 54.6 MB de texto quedan ~37 MB de parte considerativa.
# Es el comportamiento que la funcion siempre busco, pero explica por que el
# numero de chunks no es comparable uno a uno con la corrida anterior.
# =============================================================================

def build_vectorstore_limpio():
    docs = cargar_corpus_limpio(verbose=False)
    if not docs:
        print("No se cargo ningun documento")
        return None

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=900,
        chunk_overlap=120,
        separators=["\n## ", "\n---\n", "\n\n", "\n", ". ", " "],
    )
    chunks = splitter.split_documents(docs)
    print(f"\nChunks generados: {len(chunks)}")

    # ---------- red de seguridad antes de gastar GPU ----------
    malos = [c for c in chunks if any(s in c.page_content for s in CENTINELAS)]
    if malos:
        print(f"\nABORTADO: {len(malos)} chunks con marcas de basura. Ejemplo:")
        print(malos[0].page_content[:200])
        return None
    print("Verificacion: 0 chunks contaminados\n")

    print("Generando embeddings (5-10 min con T4)...")
    if os.path.exists(VECTOR_LOCAL):
        shutil.rmtree(VECTOR_LOCAL)
    os.makedirs(VECTOR_LOCAL, exist_ok=True)

    vs = Chroma.from_documents(
        documents=chunks,
        embedding=emb,
        persist_directory=VECTOR_LOCAL,
    )

    if os.path.exists(VECTOR_DRIVE):
        shutil.rmtree(VECTOR_DRIVE)
    shutil.copytree(VECTOR_LOCAL, VECTOR_DRIVE)

    print(f"\n{len(chunks)} chunks indexados")
    print("Backup guardado en Drive")
    return vs


# ---------- se reconstruye SIEMPRE: el indice viejo esta contaminado ----------
vectorstore = build_vectorstore_limpio()

# ---------- comprobacion posterior ----------
# col.get() sin limite pide todos los metadatos en una sola consulta y SQLite
# rechaza mas de 32.766 variables ("too many SQL variables"). Con ~63.000
# chunks hay que paginar.
if vectorstore is not None:
    import collections

    col = vectorstore._collection
    total = col.count()
    cats, archivos = collections.Counter(), set()
    LOTE = 5000
    for off in range(0, total, LOTE):
        meta = col.get(include=["metadatas"], limit=LOTE, offset=off)["metadatas"]
        for m in meta:
            cats[m.get("categoria")] += 1
            archivos.add(m.get("archivo"))

    print("\n" + "=" * 60)
    print(f"Embeddings en el indice : {total}")
    print(f"Documentos distintos    : {len(archivos)}")
    print("\nChunks por categoria:")
    for k, v in cats.most_common():
        print(f"   {k:<24} {v:6d}")
    print("=" * 60)


In [11]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 8, "fetch_k": 20, "lambda_mult": 0.7}
)

print("Retriever listo")

query = "autonomía de la voluntad contrato civil"
test = retriever.invoke("query: " + query)
print(f"Test: {len(test)} fragmentos recuperados\n")

for i, r in enumerate(test, 1):
    tipo = r.metadata.get("tipo", "?")
    cat = r.metadata.get("categoria", "?")
    art = r.metadata.get("num_articulo", "-")
    pag = r.metadata.get("page", "-")
    preview = r.page_content[:220].replace("\n", " ")
    print(f"{i}. {tipo} | {cat} | art={art} | pag={pag}")
    print(preview)
    print()

Retriever listo
Test: 8 fragmentos recuperados

1. sentencia | Sentencias | art=- | pag=27
passage: Radicación: 73001-31-10-002-2014-00340-01  «( ... ) la autonomía privada en cuanto libertad contractual,  comporta el razonable reconocimiento legal a toda persona de un  cúmulo de poderes o facultades proyectad

2. sentencia | Sentencias | art=- | pag=26
De ahí que, con contu ndencia, sentencie el artículo  1502 del Código Civil, que en orden a que «una persona se  obligue a otra por un acto o declaración de voluntad », resulta  indispensable que el declarante «consienta

3. sentencia | Sentencias | art=- | pag=42
si, trasciende al mero arbitrio o a la simple volición19" »  (CSJ 4902-2019, 13 nov.)  11 <<SPOTA, AG. Instituciones de Derecho Civil. Contratos. T /// Pág. 516» (referencia propia del  texto citado).

4. sentencia | Sentencias | art=- | pag=21
condición de obedecer los dictados de la razón, el sinónimo  de libertad que en ella encontró Rosseau como la posibilidad  de elegir y

In [12]:
def format_docs(docs):
    bloques = []

    for d in docs:
        tipo = d.metadata.get("tipo", "?")
        categoria = d.metadata.get("categoria", "?")
        fuente = d.metadata.get("fuente", "?")

        if tipo == "norma":
            ley = d.metadata.get("ley", "?")
            art = d.metadata.get("num_articulo", "?")
            encabezado = d.metadata.get("encabezado", "")
            cabecera = f"[NORMA | {categoria} | Ley {ley} | Artículo {art} | {fuente}]"
            if encabezado:
                cabecera += f"\n{encabezado}"
        else:
            pagina = d.metadata.get("page", "?")
            cabecera = f"[SENTENCIA | {categoria} | {fuente} | página {pagina}]"

        contenido = d.page_content.replace("passage: ", "", 1)
        bloques.append(f"{cabecera}\n{contenido}")

    return "\n\n---\n\n".join(bloques)

print("format_docs listo")

format_docs listo


In [29]:
from langchain_google_genai import ChatGoogleGenerativeAI
from google.colab import userdata

GEMINI_API_KEY = userdata.get("Gemini_API_Key")

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GEMINI_API_KEY,
    temperature=0,
)

RAG_PROMPT = ChatPromptTemplate.from_template("""
Eres un sistema de recuperación de información jurídica.
Tu única función es leer el CONTEXTO y responder con base EXCLUSIVA en él.

REGLAS ESTRICTAS:
1. SOLO puedes usar información que aparezca literalmente en el CONTEXTO.
2. Si la respuesta no está en el CONTEXTO, responde exactamente:
   "No encontrado en el corpus. El contexto recuperado no contiene información suficiente para responder."
3. PROHIBIDO usar conocimiento jurídico propio, pre-entrenamiento o inferencias externas al CONTEXTO.
4. PROHIBIDO completar, inferir o extrapolar más allá de lo que dice el texto recuperado.
5. Cita siempre el artículo, sentencia o fuente exacta que usaste del CONTEXTO.

CONTEXTO RECUPERADO:
{context}

PREGUNTA:
{question}

RESPUESTA (basada únicamente en el contexto de arriba):
""")

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

print("✅ Gemini 2.5 Flash (pago) listo — sin límite diario")

✅ Gemini 2.5 Flash (pago) listo — sin límite diario


In [30]:
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

print("Cadena RAG lista")

Cadena RAG lista


In [31]:
pregunta = "¿Qué dice la Ley 1480 sobre el derecho de retracto?"
respuesta = rag_chain.invoke("query: " + pregunta)

print("PREGUNTA:\n", pregunta)
print("\nRESPUESTA:\n", respuesta)

PREGUNTA:
 ¿Qué dice la Ley 1480 sobre el derecho de retracto?

RESPUESTA:
 Según la Ley 1480, en su Artículo 47, se entenderá pactado el derecho de retracto por parte del consumidor en todos los contratos para la venta de bienes y prestación de servicios mediante sistemas de financiación otorgada por el productor o proveedor, venta de tiempos compartidos o ventas que utilizan métodos no tradicionales o a distancia, que por su naturaleza no deban consumirse o no hayan comenzado a ejecutarse antes de cinco (5) días. En caso de hacer uso de esta facultad, el contrato se resolverá y se deberá reintegrar el dinero que el consumidor hubiese pagado.

Cita: [NORMA | Ley_1480 | Ley 1480 | Artículo 47 | ley_1480_rag]


In [32]:
pregunta = "¿Cuáles son los vicios del consentimiento?"
docs = retriever.invoke("query: " + pregunta)

print(f"Chunks recuperados: {len(docs)}\n")

for d in docs:
    print(
        "-",
        d.metadata.get("tipo"),
        "|",
        d.metadata.get("categoria"),
        "| art:",
        d.metadata.get("num_articulo", "-"),
        "| pág:",
        d.metadata.get("page", "-")
    )
    print(d.page_content[:180])
    print()

Chunks recuperados: 8

- norma | Codigo_Civil | art: 1508 | pág: -
passage: # Codigo_Civil — Extracto para RAG
> Generado el 2026-06-18 23:49

Artículo 1508
ARTÍCULO 1508. <VICIOS DEL CONSENTIMIENTO>. Los vicios de que puede adolecer el consentimi

- sentencia | Sentencias | art: - | pág: 29
afectada por irregularidades que genéricamente son denominadas en la ley y en la doctrina vicios del 
consentimiento, los cuales son error, fuerza y dolo. La fuerza o violencia es 

- norma | Codigo_Comercio | art: 107 | pág: -
passage: # Codigo_Comercio — Extracto para RAG
> Generado el 2026-06-18 23:51

Artículo 107
ARTÍCULO 107. <ERROR DE HECHO, ERROR SOBRE LA ESPECIE DE SOCIEDAD - VICIO DEL CONSENTIMI

- sentencia | Sentencias | art: - | pág: 29
extracción y preferencia de la negociación real que  se 
escondió bajo el ropaje de una declaración fingida. La 
utilidad reparadora, por su parte, aparece en la medida en 
que la 

- sentencia | Sentencias | art: - | pág: 26
De ahí que, con contu ndenc

In [33]:
class AgentState(TypedDict):
    pregunta: str
    contexto: str
    borrador: str
    feedback: str
    respuesta_final: str
    num_revisiones: int

print("Estado del agente definido")

Estado del agente definido


In [34]:
cot_prompt = ChatPromptTemplate.from_template("""
Eres un experto jurista en Derecho Contractual Civil Colombiano.
Tu tarea es responder a la PREGUNTA basándote ÚNICAMENTE en el CONTEXTO RECUPERADO.

Si recibes FEEDBACK DE AUDITORÍA, úsalo para corregir tu error anterior.

INSTRUCCIONES DE RAZONAMIENTO:
Estructura tu respuesta siguiendo este silogismo:
1. PREMISA MAYOR: Norma o jurisprudencia aplicable.
2. PREMISA MENOR: Análisis del caso/pregunta.
3. CONCLUSIÓN: Respuesta final.

CONTEXTO RECUPERADO: {contexto}
PREGUNTA: {pregunta}
FEEDBACK DE AUDITORÍA (Si existe): {feedback}

SILOGISMO JURÍDICO:
""")

cov_prompt = ChatPromptTemplate.from_template("""
Eres un Auditor Jurídico implacable. Tu tarea es evaluar la fidelidad jurídica de un BORRADOR frente a un CONTEXTO.

REGLAS:
1. ¿El borrador cita leyes o sentencias que NO están en el contexto?
2. ¿La conclusión contradice el contexto?

INSTRUCCIONES DE SALIDA:
- Si el borrador es 100% fiel al contexto, responde única y exclusivamente con:
[APROBADO]
y luego el texto exacto del borrador.
- Si detectas error, responde comenzando exactamente con:
[RECHAZADO]
y explica con claridad el problema.

CONTEXTO RECUPERADO: {contexto}
BORRADOR DE RESPUESTA: {borrador}

AUDITORÍA FINAL:
""")

print("Prompts agénticos listos")

Prompts agénticos listos


In [35]:
def nodo_recuperador(state: AgentState):
    print("Ejecutando: Recuperación de documentos...")
    docs = retriever.invoke("query: " + state["pregunta"])
    contexto_formateado = format_docs(docs)
    return {"contexto": contexto_formateado, "num_revisiones": 0, "feedback": ""}

def nodo_razonador(state: AgentState):
    print(f"Ejecutando: Agente Razonador (Intento {state['num_revisiones'] + 1})...")
    chain = cot_prompt | llm | StrOutputParser()
    borrador = chain.invoke({
        "contexto": state["contexto"],
        "pregunta": state["pregunta"],
        "feedback": state["feedback"]
    })
    return {"borrador": borrador, "num_revisiones": state["num_revisiones"] + 1}

def nodo_verificador(state: AgentState):
    print("Ejecutando: Agente Verificador (Auditoría)...")
    chain = cov_prompt | llm | StrOutputParser()
    evaluacion = chain.invoke({
        "contexto": state["contexto"],
        "borrador": state["borrador"]
    })

    if "[APROBADO]" in evaluacion:
        respuesta_limpia = evaluacion.replace("[APROBADO]", "").strip()
        return {"feedback": evaluacion, "respuesta_final": respuesta_limpia}
    else:
        return {"feedback": evaluacion}

def decidir_siguiente_paso(state: AgentState):
    if "[APROBADO]" in state["feedback"]:
        print("Auditoría superada. Finalizando.")
        return "fin"
    elif state["num_revisiones"] >= 3:
        print("Límite de revisiones alcanzado. Entregando el último borrador.")
        return "fin"
    else:
        print(f"Error detectado: {state['feedback'][:60]}... Devolviendo al Razonador.")
        return "revisar"

print("Nodos del grafo listos")

Nodos del grafo listos


In [36]:
workflow = StateGraph(AgentState)

workflow.add_node("Recuperador", nodo_recuperador)
workflow.add_node("Razonador", nodo_razonador)
workflow.add_node("Verificador", nodo_verificador)

workflow.set_entry_point("Recuperador")
workflow.add_edge("Recuperador", "Razonador")
workflow.add_edge("Razonador", "Verificador")

workflow.add_conditional_edges(
    "Verificador",
    decidir_siguiente_paso,
    {
        "fin": END,
        "revisar": "Razonador"
    }
)

app_agentica = workflow.compile()

print("Arquitectura agéntica compilada exitosamente")

Arquitectura agéntica compilada exitosamente


In [37]:
pregunta_test = "¿Cuáles son los vicios del consentimiento según la normativa colombiana?"
resultado = app_agentica.invoke({"pregunta": pregunta_test})

print("\n" + "="*50)
print("RESPUESTA FINAL DEL SISTEMA AGÉNTICO")
print("="*50)
print(resultado["respuesta_final"] if "respuesta_final" in resultado else resultado["borrador"])

Ejecutando: Recuperación de documentos...
Ejecutando: Agente Razonador (Intento 1)...
Ejecutando: Agente Verificador (Auditoría)...
Auditoría superada. Finalizando.

RESPUESTA FINAL DEL SISTEMA AGÉNTICO
BORRADOR DE RESPUESTA: SILOGISMO JURÍDICO:

1.  **PREMISA MAYOR:** Según el Artículo 1508 del Código Civil colombiano, los vicios de que puede adolecer el consentimiento son el error, la fuerza y el dolo. Esta clasificación es reiterada por la jurisprudencia de la Corte Suprema de Justicia (Sentencia SC5040-2021), que los describe como irregularidades que afectan la manifestación de voluntad, exigiendo que esta sea consciente y libre. La misma jurisprudencia define el error como la falta de correspondencia entre la representación mental del sujeto y la realidad, la fuerza como la presión física o moral que infunde miedo para obtener el consentimiento, y el dolo como todo artificio para engañar que induce o provoca un error.

2.  **PREMISA MENOR:** La pregunta solicita identificar los vi

In [38]:
def cargar_preguntas(ruta_archivo):
    print(f"Cargando preguntas desde: {ruta_archivo}")
    with open(ruta_archivo, "r", encoding="utf-8") as f:
        contenido = f.read()

    preguntas = re.findall(r'^\d+\.\s+(.*)', contenido, re.MULTILINE)
    print(f"Total de preguntas extraídas: {len(preguntas)}")
    return preguntas

print("Función cargar_preguntas lista")

Función cargar_preguntas lista


In [39]:
def ejecutar_evaluacion_comparativa(preguntas):
    resultados = []

    print("\nIniciando ejecución masiva. Esto puede tardar varios minutos...")

    for i, pregunta in enumerate(tqdm(preguntas, desc="Procesando preguntas"), 1):
        fila = {
            "ID_Pregunta": i,
            "Pregunta": pregunta,
            "Respuesta_RAG_Base": "",
            "Respuesta_Agente_CoT_CoV": "",
            "Num_Revisiones_Agente": 0,
            "Error": ""
        }

        try:
            respuesta_base = rag_chain.invoke("query: " + pregunta)
            fila["Respuesta_RAG_Base"] = respuesta_base

            time.sleep(1)

            estado_inicial = {"pregunta": pregunta, "num_revisiones": 0}
            resultado_agente = app_agentica.invoke(estado_inicial)

            if "respuesta_final" in resultado_agente:
                fila["Respuesta_Agente_CoT_CoV"] = resultado_agente["respuesta_final"]
            else:
                fila["Respuesta_Agente_CoT_CoV"] = f"[No Aprobado tras revisiones] {resultado_agente.get('borrador', '')}"

            fila["Num_Revisiones_Agente"] = resultado_agente.get("num_revisiones", 0)

        except Exception as e:
            fila["Error"] = str(e)
            print(f"\nError en la pregunta {i}: {e}")

        resultados.append(fila)

    return resultados

In [40]:
def guardar_resultados(resultados, ruta_salida):
    df = pd.DataFrame(resultados)
    df.to_csv(ruta_salida, index=False, encoding='utf-8-sig')
    print(f"\n¡Evaluación terminada! Resultados guardados en: {ruta_salida}")
    return df

print("Función guardar_resultados lista")

Función guardar_resultados lista


In [ ]:
# =============================================================================
# EVALUACION COMPARATIVA SOBRE EL CORPUS REPARADO
#
# CUIDADO: la version anterior de esta celda escribia en
#   03_Results/resultados_comparativos.csv
# que es el archivo con las 120 respuestas que calificaron los tres expertos.
# Sobrescribirlo dejaria las matrices de calificacion sin sus respuestas.
#
# Esta version escribe en un archivo aparte y se niega a pisar nada.
# =============================================================================

RUTA_GOLDEN_SET = "/content/drive/MyDrive/TG_Maestria/04_Golden_Set/golden_set.md"

# linea base intacta (corpus roto, 29.281 chunks) — NO TOCAR
RUTA_BASELINE = "/content/drive/MyDrive/TG_Maestria/03_Results/resultados_comparativos.csv"

# corrida nueva (corpus reparado, 62.784 chunks)
RUTA_RESULTADOS = "/content/drive/MyDrive/TG_Maestria/03_Results/resultados_comparativos_v2_corpus_reparado.csv"

if os.path.exists(RUTA_RESULTADOS):
    raise SystemExit(
        f"Ya existe {RUTA_RESULTADOS}.\n"
        "Renombralo o borralo a mano si quieres repetir la corrida."
    )

assert RUTA_RESULTADOS != RUTA_BASELINE, "La salida no puede ser la linea base"

lista_preguntas = cargar_preguntas(RUTA_GOLDEN_SET)
print(f"Preguntas cargadas: {len(lista_preguntas)}")
print(f"Linea base (intacta): {os.path.basename(RUTA_BASELINE)}")
print(f"Salida nueva        : {os.path.basename(RUTA_RESULTADOS)}")

# -----------------------------------------------------------------------------
# PRUEBA CORTA PRIMERO. Descomenta el lote completo solo cuando estas 3
# preguntas se vean bien: son ~90 minutos de llamadas a Gemini.
# -----------------------------------------------------------------------------
resultados_raw = ejecutar_evaluacion_comparativa(lista_preguntas[:3])

# Lote completo (las 120)
# resultados_raw = ejecutar_evaluacion_comparativa(lista_preguntas)

df_resultados = guardar_resultados(resultados_raw, RUTA_RESULTADOS)
df_resultados.head()
